[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farjanaferdausi-cs50ai/LinkedIn_Post_Generator_Agent/blob/main/LinkedIn_Post_Generator_Agent.ipynb)

<div align="center">

# 🔗 LinkedIn Post Generator Agent

**Module 21 · Ostad AI/ML Engineering Program (Batch 6)**
**Author:** Farjana Ferdausi

![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=flat-square&logo=python&logoColor=white)
![LangChain](https://img.shields.io/badge/LangChain-LCEL-1C3C3C?style=flat-square&logo=langchain&logoColor=white)
![Gemini](https://img.shields.io/badge/Google_Gemini-886FBF?style=flat-square&logo=googlegemini&logoColor=white)

</div>

### 📌 About This Notebook

Provide a **topic** and a **language** — the agent returns a publish-ready LinkedIn post (2–4 paragraphs, hook, hashtags, call to action).

> 🧠 **This is an agent, not just a prompt.**
> Instead of one LLM call, it runs a two-stage reflection pipeline:
> - ✍️ **Draft** — the LLM writes a first version.
> - 🔎 **Critique & refine** — the LLM reviews its own draft against a LinkedIn best-practices checklist and returns an improved final version.

```
topic, language ──▶ [ Draft Chain ] ──▶ [ Critique & Refine Chain ] ──▶ final post
                       (LLM call 1)            (LLM call 2)
```


## 1. Install dependencies

In [ ]:
!pip install -q -U langchain langchain-core langchain-google-genai

## 2. Set Gemini API key

Get a **free** key from
[Google AI Studio](https://aistudio.google.com/apikey).

- **In Colab:** click the 🔑 key icon in the left sidebar → "Add new secret" →
  name it `GOOGLE_API_KEY` → paste your key → toggle "Notebook access" on.
- **Anywhere else:** the cell below will just prompt you to paste it in.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Loaded GOOGLE_API_KEY from Colab secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")
    print("Loaded GOOGLE_API_KEY from manual input.")

## 3. Build the agent

Instead of writing the agent's logic directly in a notebook cell, I keep it in
a real Python module (`agent.py`). This is standard practice in production
ML engineering — notebooks are for demos and exploration, `.py` files hold the
reusable logic. Running the cell below writes `agent.py` to disk (so it's a
real file you can also open, edit, and push to GitHub separately) and then
imports it.

In [ ]:
%%writefile agent.py
"""
agent.py

An AI agent, built with LangChain + Google Gemini, that turns a
(topic, language) pair into a publish-ready LinkedIn post.

--------------------------------------------------------------------
WHY THIS COUNTS AS AN "AGENT" AND NOT JUST A SINGLE PROMPT
--------------------------------------------------------------------
A single prompt -> LLM -> output call is just a "chain." This project
instead uses a two-stage REFLECTION pattern, which is a recognized
agentic design used in real production systems:

    Stage 1 (DRAFT)    : the LLM writes a first version of the post.
    Stage 2 (CRITIQUE) : the LLM re-reads its OWN draft against a
                          checklist of LinkedIn best practices and
                          returns an improved final version.

The agent is, in effect, reasoning about and correcting its own work
before handing it back to the user - that self-correction loop is
what separates an "agent" from a plain prompt-response call.

Author: Farjana Ferdausi
Module: Ostad AI/ML Engineering Program - Module 21
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable
from langchain_google_genai import ChatGoogleGenerativeAI


# A default model. gemini-3.6-flash is on Google's free tier (no credit
# card needed) and is the current stable/GA workhorse Flash model.
#
# Note on model lifecycles: Google regularly retires older model IDs
# (e.g. gemini-2.5-flash was retired for new users shortly after this
# project started). If generate() ever raises a 404 / "model not found"
# error, check the current model list at https://ai.google.dev/gemini-api/docs/models
# and update this ONE line - no other code needs to change, which is the
# whole point of keeping the model name in a single constant.
DEFAULT_MODEL = "gemini-3.6-flash"


@dataclass
class LinkedInPost:
    """
    A small container that holds everything about one generated post.

    Keeping the draft AND the final version (instead of just the final
    text) makes it easy to show, in the demo video, exactly what the
    'critique' stage changed - this is good evidence that the agent is
    really doing multi-step reasoning, not just calling the API once.
    """
    topic: str
    language: str
    draft: str
    final: str


class LinkedInPostAgent:
    """
    Usage
    -----
    >>> agent = LinkedInPostAgent()
    >>> post = agent.generate(topic="AI in Healthcare", language="English")
    >>> print(post.final)
    """

    def __init__(self, model_name: str = DEFAULT_MODEL, temperature: float = 0.7):
        if not os.environ.get("GOOGLE_API_KEY"):
            raise ValueError(
                "GOOGLE_API_KEY is not set. Get a free key from "
                "https://aistudio.google.com/apikey and set it as an "
                "environment variable (see README.md) before creating the agent."
            )

        # ChatGoogleGenerativeAI reads the GOOGLE_API_KEY environment variable
        # automatically - this is intentional. Passing the key as an explicit
        # keyword argument is possible too, but the parameter name has changed
        # between langchain-google-genai versions; the env var is the one
        # interface Google and LangChain both guarantee stays stable.
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=temperature,
        )

        # Build both stages of the pipeline once, at start-up, so that
        # generate() just re-uses them instead of rebuilding on every call.
        self.draft_chain: Runnable = self._build_draft_chain()
        self.critique_chain: Runnable = self._build_critique_chain()

    # ------------------------------------------------------------------
    # Stage 1: DRAFT
    # ------------------------------------------------------------------
    def _build_draft_chain(self) -> Runnable:
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are a senior LinkedIn content strategist who writes "
                    "concise, professional, engaging posts. "
                    "You ALWAYS write the post entirely in the language the "
                    "user asks for - never default to English unless English "
                    "is the language requested.",
                ),
                (
                    "human",
                    "Write a LinkedIn post about: {topic}\n"
                    "Language: {language}\n\n"
                    "Requirements:\n"
                    "- 2 to 4 short paragraphs\n"
                    "- Open with a strong hook (first line must earn a click on "
                    '"see more")\n'
                    "- Use short paragraphs and line breaks, the way real "
                    "LinkedIn posts are formatted\n"
                    "- Close with a call to action or a question that invites "
                    "comments\n"
                    "- End with 3 to 5 relevant hashtags\n"
                    "- Plain text only - no markdown symbols like ** or ##",
                ),
            ]
        )
        return prompt | self.llm | StrOutputParser()

    # ------------------------------------------------------------------
    # Stage 2: CRITIQUE + REFINE
    # ------------------------------------------------------------------
    def _build_critique_chain(self) -> Runnable:
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are a strict LinkedIn editor. You improve drafts "
                    "without changing their language or their core message. "
                    "Reply with ONLY the improved post - no notes, no "
                    "preamble, no explanations.",
                ),
                (
                    "human",
                    "Topic: {topic}\n"
                    "Language: {language}\n"
                    "Draft post:\n{draft}\n\n"
                    "Review the draft against this checklist, then return the "
                    "improved final version:\n"
                    "1. Is the opening line strong enough to stop someone "
                    "mid-scroll?\n"
                    "2. Is it broken into short, easy-to-scan paragraphs?\n"
                    "3. Does it sound like a real professional, not a robot?\n"
                    "4. Are the hashtags relevant and limited to 3-5?",
                ),
            ]
        )
        return prompt | self.llm | StrOutputParser()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------
    def generate(self, topic: str, language: str = "English") -> LinkedInPost:
        """Run the full draft -> critique -> refine pipeline once."""
        if not topic or not topic.strip():
            raise ValueError("Topic cannot be empty.")
        if not language or not language.strip():
            raise ValueError("Language cannot be empty.")

        draft = self.draft_chain.invoke({"topic": topic, "language": language})
        final = self.critique_chain.invoke(
            {"topic": topic, "language": language, "draft": draft}
        )

        return LinkedInPost(topic=topic, language=language, draft=draft, final=final)


In [ ]:
from agent import LinkedInPostAgent, LinkedInPost

agent = LinkedInPostAgent()
print("Agent ready. Model:", agent.llm.model)

## 4. Try it yourself

Edit the fields below (this is a Colab **form** — no code editing needed),
then run the cell.

In [ ]:
#@title Generate a LinkedIn post
topic = "AI in Healthcare"  #@param {type:"string"}
language = "English"  #@param ["English", "Bengali", "Spanish", "Hindi", "French", "Arabic"]

result = agent.generate(topic=topic, language=language)

print("=" * 60)
print(f"TOPIC: {result.topic}   |   LANGUAGE: {result.language}")
print("=" * 60)
print(result.final)

## 5. See the agent's reasoning: draft → refined

This cell is the best one to show in the demo video — it proves the agent is
doing more than a single API call by printing **both** stages of the
pipeline.

In [ ]:
demo = agent.generate(topic="The Future of Remote Work", language="English")

print("STAGE 1 — First Draft")
print("-" * 60)
print(demo.draft)

print("\n\nSTAGE 2 — After Self-Critique & Refinement")
print("-" * 60)
print(demo.final)

## 6. Multi-language demo

The assignment specifically calls for support across languages
(English, Bengali, Spanish, ...). This cell runs the same agent on all
three example languages from the assignment brief, generating each post
only once so we don't waste API calls.

In [ ]:
demo_examples = [
    ("AI in Healthcare", "English"),
    ("Remote Work Productivity", "Bengali"),
    ("The Future of Renewable Energy", "Spanish"),
]

results = [agent.generate(topic=t, language=l) for t, l in demo_examples]

for r in results:
    print("\n" + "=" * 60)
    print(f"{r.language.upper()} — {r.topic}")
    print("=" * 60)
    print(r.final)

## 7. Save the generated posts

Exports every post from the demo above into `generated_posts.md` — useful
both as evidence of working output and as ready-to-use content for your
own LinkedIn.

In [ ]:
from datetime import datetime

def save_posts_to_markdown(posts, filename="generated_posts.md"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("# Generated LinkedIn Posts\n")
        f.write(f"*Generated on {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n")
        for p in posts:
            f.write(f"\n---\n### {p.topic} ({p.language})\n\n{p.final}\n")
    print(f"Saved {len(posts)} posts to {filename}")

save_posts_to_markdown(results)

## 8. Conclusion

In this project, I built an AI agent using LangChain and Google Gemini that
generates professional LinkedIn posts from just a topic and a target
language. Rather than a single prompt-response call, I designed the agent
around a two-stage reflection pattern — draft, then self-critique and
refine — which gives it genuine multi-step reasoning instead of a one-shot
generation.

**Possible extensions I'd explore next:**
- A tool the agent can call to pull real trending hashtags before writing.
- A tone selector (formal / casual / thought-leader).
- Wrapping this in a small Streamlit app for non-technical users.

**Tech stack:** Python, LangChain (LCEL), Google Gemini API, Google Colab.